<a href="https://colab.research.google.com/github/karlwoww/classification_of_reviews/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Dense, Embedding, Dropout, BatchNormalization
from tensorflow.keras import regularizers
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from transformers import models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=11000)

In [ ]:
print(f"Исходные размеры: train={len(train_data)}, test={len(test_data)}")


Исходные размеры: train=25000, test=25000


In [ ]:
all_data = np.concatenate([train_data, test_data])
all_labels = np.concatenate([train_labels, test_labels])

In [ ]:
indices = np.random.permutation(len(all_data))
all_data = all_data[indices]
all_labels = all_labels[indices]

In [ ]:
train_data = all_data[:49000]
train_labels = all_labels[:49000]
test_data = all_data[49000:50000]
test_labels = all_labels[49000:50000]

In [ ]:
test_data.shape

(1000,)

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

In [ ]:
def vectorize_sequences(sequences, dimension=11000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

x_train = vectorize_sequences(train_data)
x_test  = vectorize_sequences(test_data)

In [ ]:
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

In [ ]:
x_train.shape

(49000, 11000)

In [ ]:
EPOCHS = 3
BATCH_SIZE = 450

In [ ]:

model = Sequential()
model.add(Dense(64, activation='relu', input_shape=(11000,))) # 64
model.add(BatchNormalization())
model.add(Dropout(0.6))
model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))
model.add(Dense(1, activation='linear'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = model.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.13, # 0.15
    verbose=1
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
95/95 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - accuracy: 0.6971 - loss: 0.6382 - val_accuracy: 0.8653 - val_loss: 0.4340
Epoch 2/3
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 36ms/step - accuracy: 0.8847 - loss: 0.2775 - val_accuracy: 0.8986 - val_loss: 0.2997
Epoch 3/3
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - accuracy: 0.9123 - loss: 0.2261 - val_accuracy: 0.9075 - val_loss: 0.2482


In [ ]:
from sklearn.metrics import  accuracy_score
y_pred = model.predict(x_test)
y_pred_binary = (y_pred > 0.5).astype(int).flatten()
val_accuracy = history.history['val_accuracy']
print("Валидационная точность:", val_accuracy[-1])
print(f"Точность на контрольной выборке: {accuracy_score(y_test, y_pred_binary):.4f}")

Валидационная точность: 0.9075353145599365
Точность на контрольной выборке: 0.9100
